# 10 — Descarga de corridas → Drive

Resuelve los 19 accessions de `data/organismos.tsv` a corridas contra la ENA y
baja los `.sra` a `tesis/80_sra/`.

**Las sesiones de Colab se mueren, y eso es lo normal, no la excepción.** Este
notebook está hecho para eso: el estado del trabajo es qué archivos existen en
Drive, así que re-ejecutarlo retoma donde quedó. Usá `LIMITE` para que cada
sesión haga una tanda y termine.

Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]); 
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

In [ ]:
import shutil as _sh

SRA = DRIVE / '80_sra'; SRA.mkdir(parents=True, exist_ok=True)
STAGING = pathlib.Path('/content/sra_staging'); STAGING.mkdir(exist_ok=True)
MANIFIESTOS = DRIVE / '00_manifiestos'; MANIFIESTOS.mkdir(parents=True, exist_ok=True)
MANIFIESTO_DRIVE = MANIFIESTOS / 'srr_manifest.tsv'

# El ledger va a DRIVE, no adentro del clon. Dos motivos: el clon se resetea en
# cada corrida, asi que ahi adentro no sobrevive nada; y una sesion de Colab que
# se muere no puede llevarse los md5 con ella, que es exactamente lo que paso
# una vez y obligo a recalcularlos.
LEDGER_DRIVE = MANIFIESTOS / 'sra_md5.tsv'

env = dict(os.environ,
           SRA_DEST=str(SRA),
           SRA_STAGING=str(STAGING),
           SRA_LEDGER=str(LEDGER_DRIVE),
           MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'))


def _filas(p):
    return len(p.read_text().strip().split('\n')) - 1 if p.exists() else -1


# Sembrar el ledger de Drive desde la copia del repo la primera vez, y cada vez
# que el repo tenga mas filas porque commiteamos algo. Sin esto, el modo 'ledger'
# creeria que no hay nada registrado y recalcularia el md5 de las 416 corridas,
# que son ~190 GB de lectura sobre el mount de Drive.
_repo_led = CLON / 'data' / 'sra_md5.tsv'
if _filas(_repo_led) > _filas(LEDGER_DRIVE):
    _sh.copy(_repo_led, LEDGER_DRIVE)
    print('ledger sembrado desde el repo:', _filas(LEDGER_DRIVE), 'filas')

print('destino :', SRA)
print('staging :', STAGING, f'({_sh.disk_usage("/content").free/1e9:.0f} GB libres)')
print('ledger  :', LEDGER_DRIVE, f'({_filas(LEDGER_DRIVE)} filas)')


## 1. Manifiesto

Se genera con `scripts/fetch_runs.sh manifest`, que consulta la ENA y filtra a
datos de RNA: `library_source = TRANSCRIPTOMIC` —el filtro duro, porque varios
BioProjects mezclan corridas GENOMIC— y `SINGLE` para RNA-Seq, porque el PAIRED
de un proyecto de RNA-Seq no es sRNA-seq. Al final saca las corridas listadas en
`data/excluidas.tsv`.

Se guarda una copia en Drive: el clon es efímero y el manifiesto define el
trabajo pendiente. Esa copia se reusa por defecto; hay que poner
`REGENERAR = True` cuando cambió `data/organismos.tsv` o `data/excluidas.tsv`.

**La celda revisa el manifiesto que quedó en uso contra las dos specs** —que no
tenga adentro ninguna corrida excluida, y que ningún BioProject de
`organismos.tsv` quede sin corridas—. No alcanza con mirar el total: cuando se
sacó `SRR23277331` y se agregó el duplicado de `sclsc`, el conteo dio 416 antes
y después, así que un manifiesto viejo se veía idéntico a uno al día.


In [ ]:
REGENERAR = False   # True = volver a consultar la ENA y pisar la copia de Drive

import csv, subprocess, shutil as _sh

MAN_CLON = CLON / 'data' / 'srr_manifest.tsv'


def _sin_comentarios(p):
    """organismos.tsv y excluidas.tsv empiezan con comentarios `#` largos, asi
    que la primera linea del archivo NO es el encabezado."""
    return [ln for ln in p.read_text().split('\n')
            if ln.strip() and not ln.lstrip().startswith('#')]


def _excluidas():
    p = CLON / 'data' / 'excluidas.tsv'
    if not p.exists():
        return []
    return [ln.split('\t')[0] for ln in _sin_comentarios(p)[1:]]


def _proyectos_spec():
    filas = _sin_comentarios(CLON / 'data' / 'organismos.tsv')[1:]
    # columna 6 (1-indexada): bioproject. Es una fila por accession — los dos
    # BioProjects del primario de maggi son dos filas, no un campo combinado.
    return {ln.split('\t')[5].strip() for ln in filas if len(ln.split('\t')) >= 6}


def revisar():
    """Que el manifiesto en uso corresponda a la spec de AHORA, no a la de antes.

    El modo de falla que esto atrapa ya pasó: la copia de Drive era anterior a
    la exclusión de SRR23277331 y al duplicado nuevo de sclsc, §1 la reusó, y
    nada en la salida lo decía — el conteo hasta coincidía por casualidad."""
    filas = list(csv.DictReader(open(MAN_CLON), delimiter='\t'))
    corridas = {r['run'] for r in filas}
    proy_man = {r['bioproject'] for r in filas}
    print(f'manifiesto en uso: {len(filas)} corridas, {len(proy_man)} BioProjects')

    mal = [r for r in _excluidas() if r in corridas]
    faltan = sorted(_proyectos_spec() - proy_man)
    if mal:
        print('  OJO: hay corridas excluidas adentro:', ', '.join(mal))
    if faltan:
        print('  OJO: BioProjects de la spec sin ninguna corrida:', ', '.join(faltan))
    if mal or faltan:
        # Un print no alcanza: la celda seguia, las tres siguientes corrian
        # sobre el manifiesto viejo y §4 terminaba ofreciendo la fila excluida.
        # Cortar es lo unico que obliga a arreglarlo.
        raise RuntimeError(
            'el manifiesto en uso no corresponde a la spec de ahora. '
            'Poné REGENERAR = True en esta celda y corré de nuevo.')
    print('  coincide con organismos.tsv y con excluidas.tsv')


# El manifiesto de Drive es la copia buena mientras la spec no cambie. Hay que
# regenerarlo cuando cambia `data/organismos.tsv` (un BioProject nuevo) o
# `data/excluidas.tsv` (una corrida que se saca).
if MANIFIESTO_DRIVE.exists() and not REGENERAR:
    print('ya hay manifiesto en Drive; lo reuso.')
    _sh.copy(MANIFIESTO_DRIVE, MAN_CLON)
else:
    if MANIFIESTO_DRIVE.exists():
        # Guardar el anterior antes de pisarlo: si la consulta a la ENA sale
        # distinta de lo esperado, se puede comparar contra qué.
        _prev = MANIFIESTO_DRIVE.with_name('srr_manifest_prev.tsv')
        _sh.copy(MANIFIESTO_DRIVE, _prev)
        print('copia del anterior en', _prev)
    r = subprocess.run(['./scripts/fetch_runs.sh', 'manifest'], cwd=CLON,
                       capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    if r.returncode != 0:
        raise RuntimeError('fetch_runs.sh manifest fallo; no piso la copia de Drive')
    MANIFIESTO_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    _sh.copy(MAN_CLON, MANIFIESTO_DRIVE)
    print('copia guardada en', MANIFIESTO_DRIVE)

print()
revisar()


## 1b. Buscar un BioProject de reemplazo

`sclsc` es **el único de los 9 sin duplicado**: su `PRJNA985401` es RNA-Seq
PAIRED y cayó entero en el filtro. No fue un accidente — el BioProject elegido
era del tipo de experimento equivocado, y la propia nota de `organismos.tsv` ya
decía "RNA-Seq, no miRNA-Seq". Sin duplicado no hay con qué confirmar un
candidato priorizado.

De los 15 proyectos que encontró `buscar`, 13 son `RNA-Seq` —la etiqueta que se
demostró poco confiable— y quedan dos con estrategia de sRNA:

| proyecto | corridas | estrategia | spots |
| :-- | --: | :-- | --: |
| `PRJNA379694` | 6 | miRNA-Seq | 758 M |
| `PRJNA1135930` | 1 | ncRNA-Seq | 12 M |

**`PRJNA379694` da 126 M spots por corrida, que es muchísimo para miRNA-Seq** —
exactamente el perfil de una etiqueta dudosa. Por eso la celda de abajo **lo
perfila antes de adoptarlo**: `perfil` acepta un BioProject, resuelve la primera
corrida que pasa el filtro y la mide por red sin bajarla entera. Sería absurdo
reemplazar un duplicado que resultó no ser sRNA-seq por otro sin mirarlo.


In [ ]:
# Primero: perfilar los candidatos ANTES de adoptar ninguno.
CANDIDATOS = ['PRJNA379694', 'PRJNA1135930']
SPOTS      = 20000

for _prj in CANDIDATOS:
    cmd = ['./scripts/fetch_runs.sh', 'perfil', _prj, '-n', str(SPOTS)]
    print('$ ' + ' '.join(cmd))
    p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    p.wait()
    print()

# Y si querés volver a listar los proyectos de la especie:
# !cd /content/tesis && ./scripts/fetch_runs.sh buscar 'Sclerotinia sclerotiorum'


## 1c. ¿Estos datos son sRNA-seq de verdad?

`avg_len` y la etiqueta de la ENA **no alcanzan**. `SRR23277331` decía
`miRNA-Seq` y no tenía un solo read con adaptador en 40 000: era mRNA, y salió
del manifiesto.

Lo que decide es **dónde empieza el adaptador 3'**, leído junto con la longitud
del read:

| adaptador | read | veredicto |
| :-- | :-- | :-- |
| temprano, inserto en 15-50 nt | — | `PARECE sRNA-seq` |
| 0% | corto (≤50 nt) | `YA RECORTADA` — el read *es* el inserto |
| 0% | largo | `NO PARECE` — el inserto supera al read, es mRNA |
| hay, pero inserto fuera de 15-50 | — | `DUDOSA` |

La ventana es la de `fastp` (15-50 nt), no una propia: el proyecto la eligió para
no truncar tRFs (30-40 nt) ni dejar los piRNAs (24-32) sin margen.

**`TODOS = True` perfila una corrida de cada uno de los 18 proyectos.** Los 16
resueltos ya están; quedan abiertos `maggi PRJNA154615` y `phypa PRJNA277372`,
que dieron 0% con la lista de adaptadores vieja incompleta.


In [ ]:
TODOS    = True              # True = una corrida de cada proyecto (18)
CORRIDAS = ['SRR23277331']   # si TODOS=False, estas
SPOTS    = 20000

cmd = ['./scripts/fetch_runs.sh', 'perfil']
lotes = [cmd + ['--proyectos', '-n', str(SPOTS)]] if TODOS \
        else [cmd + [r, '-n', str(SPOTS)] for r in CORRIDAS]

for _c in lotes:
    print('$ ' + ' '.join(_c))
    p = subprocess.Popen(_c, cwd=CLON, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    print('exit =', p.wait())


## 2. Encolar las descargas

Recorre **toda** la cola de pendientes. `prefetch` escribe primero en el disco
de la VM, se corre `vdb-validate`, y **recién ahí** se mueve a Drive: escribir
GB directo al FUSE de Drive es lento e inestable, y un `.sra` truncado no falla
ruidosamente —alinea de menos—, así que el que no valida se descarta y nunca
llega al destino.

`HORAS` hace que corte **solo**, antes de que Colab mate la sesión a mitad de
una descarga. Re-ejecutar la celda retoma donde quedó: el estado es qué
archivos existen en Drive, no un contador.

`ORDEN` decide qué se baja primero:

- `entrenamiento` — `gadmo`, `galga` y `maggi` antes que el resto. Son los
  únicos con positivos curados por MirGeneDB, o sea de los que depende que el
  modelo sirva. Es el default.
- `chico` — organismos con menos pendientes primero, para completar organismos
  enteros cuanto antes. Un organismo completo se puede alinear; uno a medias no.
- `alfabetico` — orden fijo, útil si querés que sea predecible.

In [ ]:
ORGANISMO = ''              # '' = todos, o 'prupe', 'gadmo', ...
ORDEN     = 'entrenamiento' # entrenamiento | chico | alfabetico
HORAS     = 3               # corta solo pasadas N horas; None = sin corte
LIMITE    = None            # tope de corridas; None = la cola entera

cmd = ['./scripts/fetch_runs.sh', 'prefetch']
if ORGANISMO:
    cmd.append(ORGANISMO)
cmd += ['--orden', ORDEN]
if HORAS:
    cmd += ['--horas', str(HORAS)]
if LIMITE:
    cmd += ['-n', str(LIMITE)]

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()

## 3. Reparar el ledger

Dos cosas, las dos idempotentes:

1. **Recalcula el md5** de lo que esté bajado y no figure en el ledger. Pasa
   cuando una sesión de Colab se muere antes de que el ledger llegue a git.
2. **Saca del ledger las corridas que ya no están en el manifiesto.** Si una
   corrida se excluyó (ver `data/excluidas.tsv`), su fila vieja haría que la
   celda 4 proponga volver a commitearla — deshaciendo la exclusión sin que
   nadie lo note.

`FORMATO` aplica **solo a las que falten**: una vez movido a Drive el archivo se
llama `<RUN>.sra` venga normalizado o lite, así que del archivo no se deduce.


In [ ]:
ORGANISMO = ''           # '' = todos
FORMATO   = 'sra'        # sra | sralite — aplica a las que falten

cmd = ['./scripts/fetch_runs.sh', 'ledger']
if ORGANISMO:
    cmd.append(ORGANISMO)
cmd += ['--formato', FORMATO]

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()


## 4. Guardar el ledger

Los md5 van a git, no solo a Drive: el checksum guardado únicamente al lado del
dato no prueba nada.

El clon **es** un repo git, así que lo que falta commitear es exactamente el
`git diff` del ledger. Con 416 corridas, volcar el archivo entero serían 417
líneas para copiar por las 2 que cambiaron.


In [ ]:
import csv

en_manifiesto = {r['run'] for r in
                 csv.DictReader(open(CLON / 'data' / 'srr_manifest.tsv'), delimiter='\t')}

en_repo = set()
_repo_led = CLON / 'data' / 'sra_md5.tsv'
if _repo_led.exists():
    en_repo = set(_repo_led.read_text().strip().split('\n')[1:])

# Segundo guardia, a proposito redundante con el de §1: esta es la celda cuya
# salida se copia a git, o sea donde el error hace dano. Si el manifiesto en uso
# todavia tiene adentro una corrida excluida, no hay nada que commitear — la
# fila que ofreceria desharia la exclusion.
_excl = [ln.split('\t')[0] for ln in
         (CLON / 'data' / 'excluidas.tsv').read_text().splitlines()
         if ln.strip() and not ln.lstrip().startswith('#')][1:]
_mal = [r for r in _excl if r in en_manifiesto]
if _mal:
    raise RuntimeError(
        f'el manifiesto tiene corridas excluidas adentro: {_mal}. '
        'Es el manifiesto viejo: corré §1 con REGENERAR = True antes de esta celda. '
        'No commitees nada de lo que esta celda hubiera impreso.')

lineas = LEDGER_DRIVE.read_text().strip().split('\n') if LEDGER_DRIVE.exists() else []
# Una fila de una corrida que ya no esta en el manifiesto NO se commitea: seria
# deshacer una exclusion sin querer. Se reporta aparte.
faltan = [l for l in lineas[1:] if l not in en_repo and l.split('\t')[1] in en_manifiesto]
sobran = [l for l in lineas[1:] if l.split('\t')[1] not in en_manifiesto]

print(f'manifiesto     : {len(en_manifiesto)} corridas')
print(f'ledger en Drive: {max(len(lineas) - 1, 0)} filas')
print(f'ledger en git  : {len(en_repo)} filas')
print()
if sobran:
    print('--- en el ledger de Drive pero NO en el manifiesto (NO commitear) ---')
    for l in sobran:
        print(' ', l)
    print('  Corré la celda 3 para sacarlas del ledger de Drive.')
    print()
if faltan:
    print('--- pegale estas filas a data/sra_md5.tsv y commitealas ---')
    print('\n'.join(faltan))
elif not sobran:
    print('nada que commitear: git ya tiene todo lo que hay en Drive.')


## 5. Repetir

Volvé a correr la celda de la sección 2 hasta que `90_estado.ipynb` no muestre faltantes.
Cada tanda retoma sola, así que alcanza con re-ejecutarla.

**Colab Free no está pensado para trabajo desatendido largo**: el uso sostenido
lleva a throttling. Conviene espaciar las tandas en vez de encadenarlas.

## 6. Diagnóstico de una corrida que falla

Cuando `prefetch` sale con código 0 y aun así no deja un `.sra`, el mensaje de
la celda 2 ya muestra qué quedó en el staging. Esta celda va un paso más atrás:
pregunta al resolver de SRA qué URL devuelve para la corrida, qué dice
`vdb-dump --info` que existe, y corre `prefetch` con **la salida a la vista**.

Con eso se distinguen los dos casos que se confunden: una corrida que solo
existe en formato original (`.fastq.gz`, `.bam`, `.sff`) de una que el resolver
directamente no encuentra. La primera se puede recuperar; la segunda se excluye
del manifiesto y se declara en métodos.


In [ ]:
CORRIDAS = ['SRR317135', 'SRR1066790']   # las que fallan

cmd = ['./scripts/fetch_runs.sh', 'diag'] + CORRIDAS
print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()
